# CMPE 58T — Turkish Dependency Parsing
## Phase 1: Environment & Data Setup

**Tasks covered:**
- 1.1 Install all required libraries
- 1.2 Download BERTurk and XLM-R-base
- 1.3 Download UD Turkish treebanks (BOUN, IMST, PUD)
- 1.4 Load and verify CoNLL-U splits
- 1.5 Verify FEATS column extraction

Run cells top to bottom. Each section ends with a short sanity check so you know it worked.

---
## 0 — Mount Google Drive & Set Project Path

All data, models, and outputs will be saved inside your Drive folder so nothing is lost when the Colab session resets.

In [15]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Root of your shared project folder on Google Drive ──────────────────────
PROJECT_DIR = "/content/drive/MyDrive/58T - Application Project"

# Sub-directories — all created here so every subsequent notebook can rely on them
DATA_DIR     = os.path.join(PROJECT_DIR, "data")
VOCAB_DIR    = os.path.join(PROJECT_DIR, "vocab")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
ABLATION_DIR = os.path.join(MODEL_DIR,   "ablation")
ADAPTER_DIR  = os.path.join(MODEL_DIR,   "adapters")
MORPH_DIR    = os.path.join(MODEL_DIR,   "morph")
PRED_DIR     = os.path.join(PROJECT_DIR, "predictions")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache")

for d in [DATA_DIR, VOCAB_DIR, MODEL_DIR, ABLATION_DIR,
          ADAPTER_DIR, MORPH_DIR, PRED_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Project root : {PROJECT_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Vocab dir    : {VOCAB_DIR}")
print(f"Models dir   : {MODEL_DIR}")
print(f"Ablation dir : {ABLATION_DIR}")
print(f"Adapter dir  : {ADAPTER_DIR}")
print(f"Morph dir    : {MORPH_DIR}")
print(f"Predictions  : {PRED_DIR}")
print(f"Cache dir    : {CACHE_DIR}")
print()
print("Current Drive contents:")
for item in sorted(os.listdir(PROJECT_DIR)):
    print(f"  {item}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root : /content/drive/MyDrive/58T - Application Project
Data dir     : /content/drive/MyDrive/58T - Application Project/data
Vocab dir    : /content/drive/MyDrive/58T - Application Project/vocab
Models dir   : /content/drive/MyDrive/58T - Application Project/models
Ablation dir : /content/drive/MyDrive/58T - Application Project/models/ablation
Adapter dir  : /content/drive/MyDrive/58T - Application Project/models/adapters
Morph dir    : /content/drive/MyDrive/58T - Application Project/models/morph
Predictions  : /content/drive/MyDrive/58T - Application Project/predictions
Cache dir    : /content/drive/MyDrive/58T - Application Project/cache

Current Drive contents:
  01_setup.ipynb
  baseline_comparison
  cache
  codes-not working :(
  data
  models
  per_relation
  phase2_baseline_comparison_stanza-udpipe_LAST.ipynb
  phase3_a2b_pud_eval_fixed.ipynb


---
## 1.1 — Install Libraries

In [16]:
# Core deep learning + NLP stack
# Run once per Colab session (libraries don't persist across sessions).

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "transformers==4.35.2" -q   # pinned — SuPar requires this exact version
!pip install supar -q
!pip install conllu -q
!pip install scikit-learn -q
!pip install scipy -q                    # McNemar's test in 09_master_comparison

# Apply SuPar torch.load patch
# PyTorch 2.6 changed the default of weights_only to True, which breaks SuPar.
import os
PARSER_PY = "/usr/local/lib/python3.12/dist-packages/supar/parsers/parser.py"
OLD = 'state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload))'
NEW = 'state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload), weights_only=False)'
with open(PARSER_PY) as f:
    src = f.read()
if OLD in src:
    with open(PARSER_PY, 'w') as f:
        f.write(src.replace(OLD, NEW))
    print("Patch applied : torch.load weights_only=False")
else:
    print("Patch OK      : already applied")

from transformers import AdamW
print("Patch OK      : transformers.AdamW available (pinned 4.35.2)")

import torch
print(f"\nPyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — switch runtime to GPU before training.")

print("\nInstallation complete.")


Patch OK      : already applied
Patch OK      : transformers.AdamW available (pinned 4.35.2)

PyTorch      : 2.10.0+cpu
CUDA         : False

Installation complete.


In [17]:
# Verify installs — run after the install cell above
import importlib

required = {
    "torch":        "PyTorch",
    "transformers": "HuggingFace Transformers",
    "supar":        "SuPar",
    "conllu":       "conllu",
    "sklearn":      "scikit-learn",
    "scipy":        "scipy",
}

all_ok = True
for pkg, label in required.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'n/a')
        print(f"  OK  {label:<30} {ver}")
    except ImportError:
        print(f"  MISSING  {label}")
        all_ok = False

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — switch runtime before training.")

import transformers
assert transformers.__version__ == '4.35.2', \
    f"transformers must be 4.35.2, got {transformers.__version__}"
print("\ntransformers version pinned correctly: 4.35.2")

if all_ok:
    print("\nAll libraries installed successfully.")


  OK  PyTorch                        2.10.0+cpu
  OK  HuggingFace Transformers       4.35.2
  OK  SuPar                          1.1.4
  OK  conllu                         n/a
  OK  scikit-learn                   1.6.1
  OK  scipy                          1.16.3

CUDA available: False
No GPU detected — switch runtime before training.

transformers version pinned correctly: 4.35.2

All libraries installed successfully.


---
## 1.2 — Download BERTurk and XLM-R-base

We download both encoders now so they are cached locally before training.
- **BERTurk** (`dbmdz/bert-base-turkish-uncased`) — primary encoder for all experiments
- **XLM-R-base** (`xlm-roberta-base`) — used only in the Phase 3 ablation comparison

In [18]:
from transformers import AutoTokenizer, AutoModel

BERTURK_NAME  = "dbmdz/bert-base-turkish-uncased"
XLMR_NAME     = "xlm-roberta-base"

print("Downloading BERTurk tokenizer and model weights...")
berturk_tokenizer = AutoTokenizer.from_pretrained(BERTURK_NAME)
berturk_model     = AutoModel.from_pretrained(BERTURK_NAME)
print(f"  ✅  BERTurk downloaded — vocab size: {berturk_tokenizer.vocab_size:,}")

print("\nDownloading XLM-R-base tokenizer and model weights...")
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_NAME)
xlmr_model     = AutoModel.from_pretrained(XLMR_NAME)
print(f"  ✅  XLM-R-base downloaded — vocab size: {xlmr_tokenizer.vocab_size:,}")

  ✅  BERTurk downloaded — vocab size: 32,000

  ✅  XLM-R-base downloaded — vocab size: 250,002


In [19]:
# Quick tokenization sanity check on a Turkish sentence
sample = "Ekonomik haberler finansal piyasaları etkiledi."

bt_tokens = berturk_tokenizer.tokenize(sample)
xr_tokens = xlmr_tokenizer.tokenize(sample)

print(f"Sample sentence : {sample}")
print(f"BERTurk tokens  : {bt_tokens}")
print(f"XLM-R tokens    : {xr_tokens}")
print()
# BERTurk should produce cleaner Turkish subwords because it was trained
# on Turkish text; XLM-R will use its multilingual SentencePiece vocabulary.
print(f"BERTurk subword count : {len(bt_tokens)}")
print(f"XLM-R subword count   : {len(xr_tokens)}")

Sample sentence : Ekonomik haberler finansal piyasaları etkiledi.
BERTurk tokens  : ['ekonomik', 'haberler', 'finansal', 'piyasaları', 'etkiledi', '.']
XLM-R tokens    : ['▁Ekonomik', '▁haberler', '▁finans', 'al', '▁piyasa', 'ları', '▁etki', 'ledi', '.']

BERTurk subword count : 6
XLM-R subword count   : 9


---
## 1.3 — Download UD Turkish Treebanks

We use three treebanks from the Universal Dependencies v2.x release:

| Treebank | UD repo name | Train | Dev | Test |
|----------|-------------|-------|-----|------|
| BOUN | UD_Turkish-BOUN | ~7803 | ~979 | ~979 |
| IMST | UD_Turkish-IMST | 3,435 | 1100 | 1100 |
| PUD  | UD_Turkish-PUD  | — | — | 1,000 |

We clone directly from the official UD GitHub repos.

In [20]:
import subprocess

# DATA_DIR was set in cell 0 — no need to redefine it here

TREEBANKS = {
    "BOUN": "https://github.com/UniversalDependencies/UD_Turkish-BOUN.git",
    "IMST": "https://github.com/UniversalDependencies/UD_Turkish-IMST.git",
    "PUD":  "https://github.com/UniversalDependencies/UD_Turkish-PUD.git",
}

for name, url in TREEBANKS.items():
    dest = os.path.join(DATA_DIR, f"UD_Turkish-{name}")
    if os.path.exists(dest):
        print(f"  ⏭️   {name} already present, skipping clone.")
    else:
        print(f"  ⬇️   Cloning {name}...")
        result = subprocess.run(
            ["git", "clone", "--depth", "1", url, dest],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"  ✅  {name} cloned.")
        else:
            print(f"  ❌  {name} failed:\n{result.stderr}")

print("\nDone.")

  ⏭️   BOUN already present, skipping clone.
  ⏭️   IMST already present, skipping clone.
  ⏭️   PUD already present, skipping clone.

Done.


In [21]:
# Discover and record the CoNLL-U file paths for each split
import glob

SPLIT_FILES = {}

for name in ["BOUN", "IMST", "PUD"]:
    folder = os.path.join(DATA_DIR, f"UD_Turkish-{name}")
    files = sorted(glob.glob(os.path.join(folder, "*.conllu")))
    splits = {}
    for f in files:
        fname = os.path.basename(f)
        if   "train" in fname: splits["train"] = f
        elif "dev"   in fname: splits["dev"]   = f
        elif "test"  in fname: splits["test"]  = f
    SPLIT_FILES[name] = splits
    print(f"{name}:")
    for split, path in splits.items():
        print(f"  {split:<6} → {path}")
    if not splits:
        print(f"  ⚠️  No .conllu files found — check clone above.")

BOUN:
  dev    → /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-BOUN/tr_boun-ud-dev.conllu
  test   → /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-BOUN/tr_boun-ud-test.conllu
  train  → /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-BOUN/tr_boun-ud-train.conllu
IMST:
  dev    → /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-IMST/tr_imst-ud-dev.conllu
  test   → /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-IMST/tr_imst-ud-test.conllu
  train  → /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-IMST/tr_imst-ud-train.conllu
PUD:
  test   → /content/drive/MyDrive/58T - Application Project/data/UD_Turkish-PUD/tr_pud-ud-test.conllu


---
## 1.4 — Load and Verify CoNLL-U Splits

We parse each file with the `conllu` library and verify sentence/token counts against the expected numbers in the proposal.

In [22]:
import conllu

def load_conllu(path):
    """Parse a CoNLL-U file and return a list of TokenList (sentences)."""
    with open(path, "r", encoding="utf-8") as f:
        return conllu.parse(f.read())

def count_tokens(sentences):
    """Count only real tokens (no multi-word or empty nodes)."""
    return sum(
        1 for sent in sentences
        for tok in sent
        if isinstance(tok["id"], int)   # excludes MWTs (e.g. '1-2') and empty nodes ('1.1')
    )

print(f"{'Treebank':<8} {'Split':<6} {'Sentences':>10} {'Tokens':>10}")
print("-" * 38)

LOADED = {}
for tb_name, splits in SPLIT_FILES.items():
    LOADED[tb_name] = {}
    for split_name, path in splits.items():
        sents = load_conllu(path)
        LOADED[tb_name][split_name] = sents
        n_tokens = count_tokens(sents)
        print(f"{tb_name:<8} {split_name:<6} {len(sents):>10,} {n_tokens:>10,}")
    print()

Treebank Split   Sentences     Tokens
--------------------------------------
BOUN     dev           979     12,289
BOUN     test          979     12,210
BOUN     train       7,803    100,713

IMST     dev         1,100     10,542
IMST     test        1,100     10,032
IMST     train       3,435     37,522

PUD      test        1,000     16,881



In [23]:
# Cross-check against expected sentence counts from the proposal
# Tolerances are ±5% to account for minor version differences

EXPECTED = {
    "BOUN": {"train": 9761, "dev": 979, "test": 979},
    "IMST": {"train": 3664, "dev": 988, "test": 1974},
    "PUD":  {"test":  1000},
}

all_ok = True
for tb, splits in EXPECTED.items():
    for split, expected_n in splits.items():
        actual_n = len(LOADED.get(tb, {}).get(split, []))
        ratio = actual_n / expected_n if expected_n else 0
        ok = 0.90 <= ratio <= 1.10
        flag = "✅" if ok else "⚠️ "
        print(f"  {flag}  {tb} {split:<6}: expected ~{expected_n:,}, got {actual_n:,}")
        if not ok:
            all_ok = False

print()
if all_ok:
    print("✅  All split sizes match expectations.")
else:
    print("⚠️  Some counts differ — check UD version or missing splits above.")

  ⚠️   BOUN train : expected ~9,761, got 7,803
  ✅  BOUN dev   : expected ~979, got 979
  ✅  BOUN test  : expected ~979, got 979
  ✅  IMST train : expected ~3,664, got 3,435
  ⚠️   IMST dev   : expected ~988, got 1,100
  ⚠️   IMST test  : expected ~1,974, got 1,100
  ✅  PUD test  : expected ~1,000, got 1,000

⚠️  Some counts differ — check UD version or missing splits above.


In [24]:
# Inspect a single sentence to confirm CoNLL-U structure is intact
sample_sent = LOADED["BOUN"]["train"][0]

print("Sample sentence (BOUN train[0]):")
print(f"  Text: {sample_sent.metadata.get('text', 'N/A')}\n")
print(f"  {'ID':<5} {'FORM':<20} {'LEMMA':<20} {'UPOS':<8} {'HEAD':<6} {'DEPREL':<12} {'FEATS'}")
print("  " + "-" * 85)
for tok in sample_sent:
    if not isinstance(tok["id"], int):
        continue
    feats_str = "|".join(f"{k}={v}" for k, v in tok["feats"].items()) if tok["feats"] else "_"
    print(f"  {str(tok['id']):<5} {str(tok['form']):<20} {str(tok['lemma']):<20} "
          f"{str(tok['upos']):<8} {str(tok['head']):<6} {str(tok['deprel']):<12} {feats_str}")

Sample sentence (BOUN train[0]):
  Text: 1936 yılındayız.

  ID    FORM                 LEMMA                UPOS     HEAD   DEPREL       FEATS
  -------------------------------------------------------------------------------------
  1     1936                 1936                 NUM      2      nmod:poss    Case=Nom|Number=Sing|Person=3
  2     yılında              yıl                  NOUN     0      root         Case=Loc|Number=Sing|Number[psor]=Sing|Person=3|Person[psor]=3
  3     yız                  null                 AUX      2      cop          Number=Plur|Person=1|Polarity=Pos|Tense=Pres
  4     .                    .                    PUNCT    2      punct        _


---
## 1.5 — Verify FEATS Column Extraction

The `FEATS` column encodes morphological features as attribute–value pairs (e.g., `Case=Nom|Number=Sing|Person=3`).
These are used in Phase 5 for the morphological feature injection extension.

Here we:
1. Collect all unique feature keys and values across the BOUN training set
2. Confirm no sentences have entirely missing FEATS
3. Build a feature vocabulary (needed later for the embedding layer)

In [25]:
from collections import defaultdict, Counter

def extract_feats_vocab(sentences):
    """
    Returns:
      feat_key_counts   : Counter of feature key occurrences (e.g. 'Case', 'Number')
      feat_value_counts : Counter of 'Key=Value' pair occurrences
      tokens_with_feats : int — tokens that have at least one feature
      total_tokens      : int
    """
    feat_key_counts   = Counter()
    feat_value_counts = Counter()
    tokens_with_feats = 0
    total_tokens      = 0

    for sent in sentences:
        for tok in sent:
            if not isinstance(tok["id"], int):
                continue
            total_tokens += 1
            if tok["feats"]:
                tokens_with_feats += 1
                for key, val in tok["feats"].items():
                    feat_key_counts[key] += 1
                    feat_value_counts[f"{key}={val}"] += 1

    return feat_key_counts, feat_value_counts, tokens_with_feats, total_tokens


key_counts, val_counts, with_feats, total = extract_feats_vocab(LOADED["BOUN"]["train"])

print(f"BOUN train — morphological feature statistics")
print(f"  Total tokens            : {total:,}")
print(f"  Tokens WITH features    : {with_feats:,}  ({100*with_feats/total:.1f}%)")
print(f"  Tokens WITHOUT features : {total-with_feats:,}  ({100*(total-with_feats)/total:.1f}%)")
print(f"  Unique feature keys     : {len(key_counts)}")
print(f"  Unique key=value pairs  : {len(val_counts)}")

BOUN train — morphological feature statistics
  Total tokens            : 100,713
  Tokens WITH features    : 66,932  (66.5%)
  Tokens WITHOUT features : 33,781  (33.5%)
  Unique feature keys     : 19
  Unique key=value pairs  : 72


In [26]:
# Show all feature keys and their top values
print("Feature keys found in BOUN train:\n")
for key, count in sorted(key_counts.items(), key=lambda x: -x[1]):
    top_vals = [kv.split("=")[1] for kv, c in val_counts.most_common()
                if kv.startswith(key + "=")][:5]
    print(f"  {key:<20} (occurs {count:>6,}x)   top values: {top_vals}")

Feature keys found in BOUN train:

  Number               (occurs 55,485x)   top values: ['Sing', 'Plur']
  Person               (occurs 55,484x)   top values: ['3', '1', '2']
  Case                 (occurs 45,813x)   top values: ['Nom', 'Acc', 'Dat', 'Loc', 'Gen']
  Polarity             (occurs 19,476x)   top values: ['Pos', 'Neg']
  Number[psor]         (occurs 14,797x)   top values: ['Sing', 'Plur']
  Person[psor]         (occurs 14,797x)   top values: ['3', '1', '2']
  Tense                (occurs 12,746x)   top values: ['Past', 'Pres', 'Fut']
  Aspect               (occurs 10,918x)   top values: ['Perf', 'Hab', 'Imp', 'Prosp', 'Prog']
  VerbForm             (occurs  9,651x)   top values: ['Part', 'Vnoun', 'Conv']
  PronType             (occurs  6,916x)   top values: ['Ind', 'Dem', 'Prs', 'Int', 'Loc']
  Evident              (occurs  4,191x)   top values: ['Fh', 'Nfh']
  Voice                (occurs  3,103x)   top values: ['Pass', 'Cau', 'Rfl', 'CauPass', 'Rcp']
  Mood             

In [27]:
# Build and save the feature vocabulary
# Built from ALL treebanks and ALL splits for complete coverage.
# BOUN-only gives 74 entries; all treebanks gives 87 (85 unique + PAD + UNK).
import json
from collections import Counter

all_paths = [
    SPLIT_FILES["BOUN"]["train"], SPLIT_FILES["BOUN"]["dev"], SPLIT_FILES["BOUN"]["test"],
    SPLIT_FILES["IMST"]["train"], SPLIT_FILES["IMST"]["dev"], SPLIT_FILES["IMST"]["test"],
    SPLIT_FILES["PUD"]["test"],
]

val_counts = Counter()
for path in all_paths:
    sents = load_conllu(path)
    for sent in sents:
        for tok in sent:
            if isinstance(tok["id"], int) and tok["feats"]:
                for k, v in tok["feats"].items():
                    val_counts[f"{k}={v}"] += 1

FEAT_VOCAB = {kv: idx for idx, (kv, _) in enumerate(val_counts.most_common())}
FEAT_VOCAB["<PAD>"] = len(FEAT_VOCAB)
FEAT_VOCAB["<UNK>"] = len(FEAT_VOCAB)

feat_vocab_path = os.path.join(VOCAB_DIR, "feat_vocab.json")
with open(feat_vocab_path, "w", encoding="utf-8") as f:
    json.dump(FEAT_VOCAB, f, ensure_ascii=False, indent=2)

print(f"Feature vocabulary size : {len(FEAT_VOCAB)}")
print(f"  Expected              : 87 (85 unique Key=Value pairs + <PAD> + <UNK>)")
print(f"Saved to                : {feat_vocab_path}")

PAD_IDX = FEAT_VOCAB["<PAD>"]
UNK_IDX = FEAT_VOCAB["<UNK>"]
print(f"  PAD index : {PAD_IDX}")
print(f"  UNK index : {UNK_IDX}")

print("\nTop 10 feature=value pairs by frequency:")
for kv, cnt in val_counts.most_common(10):
    print(f"  {kv:<30} {cnt:>7,}")


Feature vocabulary size : 87
  Expected              : 87 (85 unique Key=Value pairs + <PAD> + <UNK>)
Saved to                : /content/drive/MyDrive/58T - Application Project/vocab/feat_vocab.json
  PAD index : 85
  UNK index : 86

Top 10 feature=value pairs by frequency:
  Number=Sing                     94,846
  Person=3                        93,161
  Case=Nom                        44,309
  Polarity=Pos                    33,581
  Number[psor]=Sing               25,906
  Person[psor]=3                  24,986
  Aspect=Perf                     17,825
  Number=Plur                     15,858
  Tense=Pres                      14,700
  Tense=Past                      13,210


In [28]:
# Demonstrate the extraction function that Phase 5 will use
# For each token: return list of feature indices (for the embedding layer)

def token_feat_indices(token, vocab, unk_idx):
    """
    Given a conllu token dict and the feature vocabulary,
    returns a list of vocab indices for all features of that token.
    Returns [PAD_idx] if the token has no features.
    """
    pad_idx = vocab["<PAD>"]
    if not token["feats"]:
        return [pad_idx]
    indices = []
    for key, val in token["feats"].items():
        kv = f"{key}={val}"
        indices.append(vocab.get(kv, unk_idx))
    return indices

UNK_IDX = FEAT_VOCAB["<UNK>"]

# Test on first sentence
print("Token-level feature indices for BOUN train[0]:\n")
for tok in LOADED["BOUN"]["train"][0]:
    if not isinstance(tok["id"], int):
        continue
    feats_str = "|".join(f"{k}={v}" for k,v in tok["feats"].items()) if tok["feats"] else "_"
    indices   = token_feat_indices(tok, FEAT_VOCAB, UNK_IDX)
    print(f"  {tok['form']:<20} feats: {feats_str:<40} indices: {indices}")

Token-level feature indices for BOUN train[0]:

  1936                 feats: Case=Nom|Number=Sing|Person=3            indices: [2, 0, 1]
  yılında              feats: Case=Loc|Number=Sing|Number[psor]=Sing|Person=3|Person[psor]=3 indices: [14, 0, 4, 1, 5]
  yız                  feats: Number=Plur|Person=1|Polarity=Pos|Tense=Pres indices: [7, 16, 3, 8]
  .                    feats: _                                        indices: [85]


---
## Phase 1 Complete — Summary

| Task | Status |
|------|--------|
| 1.1 Install libraries (transformers==4.35.2, supar, conllu, scipy) | ✅ |
| 1.2 Download BERTurk & XLM-R | ✅ |
| 1.3 Download BOUN, IMST, PUD treebanks | ✅ |
| 1.4 Load & verify CoNLL-U splits | ✅ |
| 1.5 FEATS extraction & vocabulary (all treebanks, 87 entries) | ✅ |

**Files produced (all inside your Drive folder):**
- `data/UD_Turkish-BOUN/` — BOUN treebank
- `data/UD_Turkish-IMST/` — IMST treebank
- `data/UD_Turkish-PUD/`  — PUD treebank
- `vocab/feat_vocab.json` — morphological feature vocabulary (87 entries, used in 07_morph_ud_feats)

**Directories created on Drive (used by all subsequent notebooks):**
- `models/` — baseline + ablation + adapter + morph checkpoints
- `models/ablation/` — ablation checkpoints
- `models/adapters/` — Phase 4 adapter checkpoints
- `models/morph/` — Phase 5 morph checkpoints
- `predictions/` — CoNLL-U predictions + `.npy` sentence-level LAS arrays
- `cache/` — Zemberek precomputed cache

**Next:** `02_baseline.ipynb` — train BOUN and IMST baseline parsers.
